# Przykład 2: Q-learning — parkowanie / dojazd do celu w 1D

Ten przykład jest bezpośrednią analogią ćwiczenia z `path_follower`: tam robot ma
ciągłą obserwację (kąt do punktu trasy), którą trzeba zamienić na dyskretny stan, a akcje
to skończony zestaw prędkości wykonywanych co 100 ms. Tu robimy to samo, ale na prostej osi.

**Poprzedni notebook (korytarz)** miał stan dostępny od razu: `stan = pozycja_na_planszy`.
**Tutaj** najpierw powstaje obserwacja ciągła, a dopiero potem indeks:

```
e = x_cel - x_robot          (ciągłe, ze znakiem)
stan = dyskretyzuj(e)        (indeks wiersza tablicy Q)
```

Sam algorytm Q-learningu (wybór akcji, aktualizacja, pętla uczenia) jest **taki sam jak
w korytarzu** — zmienia się tylko sposób tworzenia stanu.

| Element | Znaczenie |
|---|---|
| obserwacja | błąd położenia `e = x_cel - x_robot` [m]; `e > 0`: cel po prawej, `e < 0`: po lewej |
| akcje | prędkość `v` ze skończonego zbioru, np. `{-1, -0.5, 0, 0.5, 1}` [m/s] |
| dynamika | `x(t+1) = x(t) + v · Δt` (bez bezwładności: prędkość obowiązuje cały krok) |
| sukces | `|e| < TOLERANCE` |
| nagroda | `+10` za sukces, w pozostałych krokach `-|e| - 0.01` |

W każdym epizodzie losowane są zarówno pozycja robota, jak i pozycja celu. Dzięki temu,
że stanem jest błąd `e`, ta sama polityka działa dla celu po lewej i po prawej.

Uwaga: fizycznie to uproszczony „dojazd do celu 1D" (nie modelujemy hamowania), a nazwa
„parkowanie" jest tylko ilustracją.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

## 1. Parametry (to można zmieniać i testować dla różnych wartości)

In [ ]:
# --- środowisko ---
X_MIN, X_MAX = -5.0, 5.0     # zakres pozycji robota i celu [m]
DELTA_T = 0.2                # czas między decyzjami agenta [s]
TOLERANCE = 0.1              # sukces, gdy |e| < TOLERANCE [m]
MIN_START_DISTANCE = 0.5     # minimalna początkowa odległość od celu, > TOLERANCE [m]
MAX_STEPS = 300              # limit kroków w jednym epizodzie [kroki]

# --- akcje ---
SPEEDS = np.array([-1.0, -0.5, 0.0, 0.5, 1.0])   # dostępne prędkości [m/s]

# --- dyskretyzacja stanu ---
E_MIN, E_MAX = X_MIN - X_MAX, X_MAX - X_MIN      # zakres błędu e: [-10, 10] [m]
N_BINS = 100                                     # liczba przedziałów błędu [-]

# --- nagrody ---
REWARD_GOAL = 10.0           # nagroda za osiągnięcie celu [-]
STEP_PENALTY = 0.01          # stała kara za każdy krok [-]

# --- Q-learning ---
N_EPISODES = 3000            # liczba epizodów uczenia [-]
ALPHA = 0.2                  # współczynnik uczenia, (0, 1] [-]
GAMMA = 0.95                 # współczynnik dyskontowania, (0, 1] [-]

# --- strategia epsilon-zachłanna ---
EPSILON_START = 1.0          # epsilon w pierwszym epizodzie, [0, 1] [-]
EPSILON_END = 0.05           # epsilon w ostatnim epizodzie, [0, 1] [-]
EPSILON_MODE = "exp"         # "const", "linear" lub "exp"

SEED = 0                     # ziarno generatora losowego (powtarzalność wyników)

## 2. Środowisko

Środowisko zwraca **ciągłą** obserwację `e`. Algorytm uczący nie zna równania ruchu.

In [ ]:
class ParkingEnv:
    """Dojazd do celu na osi. Obserwacją jest błąd położenia e = x_cel - x_robot [m]."""

    def __init__(self, x_min, x_max, delta_t, tolerance, min_start_distance, speeds):
        self.x_min, self.x_max = x_min, x_max
        self.delta_t = delta_t                       # [s]
        self.tolerance = tolerance                   # [m]
        self.min_start_distance = min_start_distance # [m]
        self.speeds = np.asarray(speeds, dtype=float)  # [m/s]
        self.x_robot = 0.0                           # pozycja robota [m]
        self.x_goal = 0.0                            # pozycja celu [m]

    def reset(self, rng, x_robot=None, x_goal=None):
        """Losuje pozycje robota i celu (lub przyjmuje zadane). Zwraca e [m]."""
        if x_robot is not None and x_goal is not None:
            self.x_robot, self.x_goal = float(x_robot), float(x_goal)
        else:
            while True:   # losujemy do skutku, by start nie był już "w celu"
                self.x_robot = rng.uniform(self.x_min, self.x_max)
                self.x_goal = rng.uniform(self.x_min, self.x_max)
                if abs(self.x_goal - self.x_robot) >= self.min_start_distance:
                    break
        return self.x_goal - self.x_robot

    def step(self, action):
        """Wykonuje akcję (indeks prędkości). Zwraca (e, nagroda, czy_terminalny)."""
        if not 0 <= action < len(self.speeds):
            raise ValueError(f"Niepoprawny numer akcji: {action}")
        v = self.speeds[action]                                   # [m/s]
        self.x_robot += v * self.delta_t                          # [m]
        e = self.x_goal - self.x_robot                            # [m]
        if abs(e) < self.tolerance:
            return e, REWARD_GOAL, True
        return e, -abs(e) - STEP_PENALTY, False


def make_env(delta_t=None, speeds=None):
    return ParkingEnv(X_MIN, X_MAX, DELTA_T if delta_t is None else delta_t, TOLERANCE,
                      MIN_START_DISTANCE, SPEEDS if speeds is None else speeds)


def draw_axis(ax, x_robot, x_goal, title=""):
    """Rysuje oś z robotem (niebieski) i celem (czerwona gwiazdka)."""
    ax.axhline(0, color="gray", lw=1)
    ax.plot([x_goal], [0], "r*", ms=18, label="cel")
    ax.plot([x_robot], [0], "bs", ms=14, label="robot")
    ax.set(xlim=(X_MIN - 0.5, X_MAX + 0.5), ylim=(-1, 1), xlabel="x [m]", title=title)
    ax.set_yticks([])
    ax.legend(loc="upper right")


fig, ax = plt.subplots(figsize=(8, 1.8))
env_demo = make_env()
e0 = env_demo.reset(np.random.default_rng(SEED))
draw_axis(ax, env_demo.x_robot, env_demo.x_goal, f"Przykładowy początek epizodu: e = {e0:+.2f} m")
plt.show()

## 3. Reprezentacja stanu: dyskretyzacja błędu

Tablica Q ma skończoną liczbę wierszy, a `e` jest liczbą rzeczywistą. Dzielimy zakres
`[E_MIN, E_MAX]` na `N_BINS` równych przedziałów o szerokości
`(E_MAX - E_MIN) / N_BINS` (domyślnie 0,2 m). Wartości spoza zakresu są **przycinane**
do skrajnego przedziału. To jedyna część, której nie było w korytarzu.

In [ ]:
def discretize_error(e, n_bins=None, e_min=None, e_max=None):
    """Zamienia ciągłe e [m] na indeks przedziału 0 … n_bins-1."""
    n_bins = N_BINS if n_bins is None else n_bins
    e_min = E_MIN if e_min is None else e_min
    e_max = E_MAX if e_max is None else e_max
    if not np.isfinite(e):
        raise ValueError("Obserwacja musi być liczbą skończoną")
    width = (e_max - e_min) / n_bins
    index = int(np.floor((e - e_min) / width))
    return int(np.clip(index, 0, n_bins - 1))                   # przycięcie do skrajnych przedziałów


def bin_edges(index, n_bins=None, e_min=None, e_max=None):
    """Granice [dolna, górna) przedziału o danym indeksie [m]."""
    n_bins = N_BINS if n_bins is None else n_bins
    e_min = E_MIN if e_min is None else e_min
    e_max = E_MAX if e_max is None else e_max
    width = (e_max - e_min) / n_bins
    return e_min + index * width, e_min + (index + 1) * width


print(f"Szerokość przedziału: {(E_MAX - E_MIN) / N_BINS:.2f} m, liczba stanów: {N_BINS}\n")
print("   e [m] | indeks stanu | granice przedziału [m]")
for e in [-12.0, -10.0, -3.14, -0.25, -0.05, 0.0, 0.05, 0.25, 3.14, 9.99, 15.0]:
    s = discretize_error(e)
    lo, hi = bin_edges(s)
    print(f"{e:8.2f} | {s:12d} | [{lo:+.2f}, {hi:+.2f})")

### Wizualizacja przedziałów
Pionowe linie to granice przedziałów. Wszystkie wartości `e` między dwiema liniami są dla
agenta **nierozróżnialne** — mają ten sam stan i tę samą politykę. Zwróć uwagę, jak gęsto
leżą linie w pobliżu zera, gdzie precyzja ma największe znaczenie.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2.2))
edges = np.linspace(E_MIN, E_MAX, N_BINS + 1)
for edge in edges:
    ax.axvline(edge, color="gray", lw=0.4)
ax.axvspan(-TOLERANCE, TOLERANCE, color="green", alpha=0.3, label=f"|e| < {TOLERANCE} m (sukces)")
sample = np.array([-7.3, -2.05, 0.37, 4.61])
ax.plot(sample, np.zeros_like(sample), "ro", label="przykładowe e")
for e in sample:
    ax.annotate(f"stan {discretize_error(e)}", (e, 0), xytext=(0, 12), textcoords="offset points",
                ha="center")
ax.set(xlabel="błąd e = x_cel - x_robot [m]", yticks=[], title="Dyskretyzacja błędu położenia")
ax.legend(loc="lower center", ncol=2)
plt.show()

## 4. Wybór akcji, aktualizacja Q, pętla uczenia

Poniższe funkcje są **takie same jak w notebooku z korytarzem**. Jedyna różnica:
w pętli stan nie jest bezpośrednio obserwacją, tylko `discretize_error(e)`.

In [ ]:
def choose_action(q_table, state, epsilon, rng):
    """Strategia epsilon-zachłanna z losowym rozstrzyganiem remisów."""
    n_actions = q_table.shape[1]
    if rng.random() < epsilon:
        return int(rng.integers(n_actions))
    best = np.flatnonzero(q_table[state] == q_table[state].max())
    return int(rng.choice(best))


def update_q_table(q_table, state, action, reward, next_state, terminal, alpha, gamma):
    """Jedna aktualizacja Q-learningu dla przejścia (s, a, r, s')."""
    if terminal:
        target = reward                                           # brak następnego kroku
    else:
        target = reward + gamma * np.max(q_table[next_state])
    q_table[state, action] += alpha * (target - q_table[state, action])


def epsilon_for_episode(episode, n_episodes, eps_start, eps_end, mode):
    """Wartość epsilon w danym epizodzie (numerowanym od 0)."""
    if mode == "const":
        return eps_start
    progress = episode / max(1, n_episodes - 1)
    if mode == "linear":
        return eps_start + (eps_end - eps_start) * progress
    if mode == "exp":
        return eps_start * (eps_end / eps_start) ** progress
    raise ValueError(f"Nieznany tryb epsilon: {mode}")


def train(env, encode_state, n_states, n_actions, n_episodes, alpha, gamma,
          eps_start, eps_end, eps_mode, max_steps, seed):
    """Uczy agenta. Zwraca tablicę Q i historię uczenia (po jednej wartości na epizod)."""
    rng = np.random.default_rng(seed)
    q_table = np.zeros((n_states, n_actions))
    history = {"return": [], "steps": [], "success": [], "epsilon": []}

    for episode in range(n_episodes):
        epsilon = epsilon_for_episode(episode, n_episodes, eps_start, eps_end, eps_mode)
        state = encode_state(env.reset(rng))
        total_reward, terminal, steps = 0.0, False, 0

        while not terminal and steps < max_steps:
            action = choose_action(q_table, state, epsilon, rng)
            observation, reward, terminal = env.step(action)
            next_state = encode_state(observation)
            # Limit kroków NIE jest stanem terminalnym, więc nadal liczymy max Q(s', .).
            update_q_table(q_table, state, action, reward, next_state, terminal, alpha, gamma)
            state = next_state
            total_reward += reward
            steps += 1

        history["return"].append(total_reward)
        history["steps"].append(steps)
        history["success"].append(terminal)
        history["epsilon"].append(epsilon)
    return q_table, history


def run_greedy_episode(env, encode_state, q_table, max_steps, rng, x_robot=None, x_goal=None):
    """Epizod bez eksploracji (epsilon = 0). Zwraca ślad {t, x, x_goal, e, v, r} i sukces."""
    e = env.reset(rng, x_robot, x_goal)
    state = encode_state(e)
    trace = dict(t=[0.0], x=[env.x_robot], x_goal=[env.x_goal], e=[e], v=[], r=[])
    terminal = False
    for k in range(1, max_steps + 1):
        action = choose_action(q_table, state, 0.0, rng)
        e, reward, terminal = env.step(action)
        state = encode_state(e)
        trace["t"].append(k * env.delta_t)
        trace["x"].append(env.x_robot)
        trace["x_goal"].append(env.x_goal)
        trace["e"].append(e)
        trace["v"].append(float(env.speeds[action]))
        trace["r"].append(reward)
        if terminal:
            break
    return trace, terminal


def success_rate(env, encode_state, q_table, n_episodes, max_steps, seed):
    """Odsetek udanych epizodów w trybie zachłannym (epsilon = 0) i średnia liczba kroków."""
    rng = np.random.default_rng(seed)
    results = [run_greedy_episode(env, encode_state, q_table, max_steps, rng) for _ in range(n_episodes)]
    ok = np.array([r[1] for r in results])
    steps = np.array([len(r[0]["v"]) for r in results])
    return ok.mean(), (steps[ok].mean() if ok.any() else float("nan"))

## 5. Początkowa tablica Q
Wymiary: `liczba przedziałów błędu × liczba prędkości`.

In [ ]:
env = make_env()
q_init = np.zeros((N_BINS, len(SPEEDS)))
print(f"Tablica Q: {q_init.shape[0]} stanów × {q_init.shape[1]} akcji = {q_init.size} wartości")
print("Akcje (prędkości [m/s]):", SPEEDS)
print("Pierwsze 3 wiersze:\n", q_init[:3])

## 6. Trening

In [ ]:
encode = discretize_error
q_table, history = train(env, encode, N_BINS, len(SPEEDS), N_EPISODES, ALPHA, GAMMA,
                         EPSILON_START, EPSILON_END, EPSILON_MODE, MAX_STEPS, SEED)
print(f"Skuteczność w ostatnich 200 epizodach (z eksploracją): {np.mean(history['success'][-200:]):.0%}")
print(f"Odwiedzone stany: {int(np.sum(np.any(q_table != 0, axis=1)))} z {N_BINS}")

### Co zobaczyć na wykresach
* **Suma nagród** (ujemna, bo większość kroków ma karę `-|e|`) powinna rosnąć ku zeru:
  im szybciej robot dojeżdża, tym mniej kar zbiera.
* **Czas dojazdu** (`kroki × Δt`) powinien maleć; wartość 300 kroków oznacza epizod bez sukcesu.
* **Odsetek sukcesów** (średnia krocząca) powinien rosnąć do wartości bliskiej 100%.
* Epizody są losowe (inne pozycje startowe), więc wykres jest szumiący — stąd średnia krocząca.

In [ ]:
def moving_average(values, window=50):
    values = np.asarray(values, dtype=float)
    if len(values) < window:
        return np.arange(len(values)), values
    ma = np.convolve(values, np.ones(window) / window, mode="valid")
    return np.arange(len(values) - len(ma), len(values)), ma


fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(history["return"], alpha=0.25, label="pojedynczy epizod")
axes[0].plot(*moving_average(history["return"]), label="średnia krocząca (50)")
axes[0].set(xlabel="numer epizodu [-]", ylabel="suma nagród [-]", title="Suma nagród w epizodzie")
axes[0].legend()

time_to_goal = np.array(history["steps"]) * DELTA_T
axes[1].plot(time_to_goal, alpha=0.25, label="pojedynczy epizod")
axes[1].plot(*moving_average(time_to_goal), label="średnia krocząca (50)")
axes[1].set(xlabel="numer epizodu [-]", ylabel="czas epizodu [s]", title="Czas do końca epizodu")
axes[1].legend()

axes[2].plot(*moving_average(np.array(history["success"], dtype=float)), label="odsetek sukcesów")
axes[2].plot(history["epsilon"], color="tab:red", label="epsilon")
axes[2].set(xlabel="numer epizodu [-]", ylabel="wartość [-]", title="Sukcesy i eksploracja")
axes[2].legend()
plt.tight_layout()
plt.show()

## 7. Wyuczona polityka w funkcji błędu

Dla środka każdego przedziału pokazujemy akcję `argmax_a Q(s, a)`. Oczekujemy:
* `e > 0` (cel po prawej) → prędkość dodatnia,
* `e < 0` (cel po lewej) → prędkość ujemna,
* blisko `e ≈ 0` → łagodniejsze prędkości (hamowanie przy celu).

Szare punkty to stany, w których tablica Q nigdy nie została zaktualizowana (agent tam nie bywał),
więc polityka w nich jest przypadkowa.

In [ ]:
def greedy_policy(q_table):
    """Najlepsza prędkość [m/s] dla każdego stanu oraz maska stanów odwiedzonych."""
    visited = np.any(q_table != 0, axis=1)
    best_idx = np.array([np.argmax(row) for row in q_table])
    return SPEEDS[best_idx], visited


centers = np.array([sum(bin_edges(s)) / 2 for s in range(N_BINS)])
policy_speed, visited = greedy_policy(q_table)

fig, ax = plt.subplots(figsize=(10, 4))
ax.step(centers[visited], policy_speed[visited], where="mid", color="tab:blue", alpha=0.4)
ax.plot(centers[visited], policy_speed[visited], "o", color="tab:blue", label="wybrana prędkość")
ax.plot(centers[~visited], policy_speed[~visited], "o", color="lightgray", label="stan nieodwiedzony")
ax.axvspan(-TOLERANCE, TOLERANCE, color="green", alpha=0.2, label="strefa sukcesu")
ax.set(xlabel="błąd e = x_cel - x_robot [m]", ylabel="wybrana prędkość v [m/s]",
       title="Polityka: prędkość w zależności od odległości do celu")
ax.legend()
plt.show()

fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(q_table.T, aspect="auto", cmap="viridis", origin="lower",
               extent=[E_MIN, E_MAX, -0.5, len(SPEEDS) - 0.5])
ax.set_yticks(range(len(SPEEDS)))
ax.set_yticklabels([f"{v:+.1f}" for v in SPEEDS])
ax.set(xlabel="błąd e [m] (przedział stanu)", ylabel="prędkość v [m/s]", title="Tablica Q")
fig.colorbar(im, label="Q [-]")
plt.show()

## 8. Demonstracyjne epizody bez eksploracji (epsilon = 0)

Cel po prawej i po lewej od robota. Na wykresach: pozycja robota `x(t)`, pozycja celu
`x_cel`, błąd `e(t)` oraz wybierana prędkość. Szukaj tego, czy `e(t)` zbliża się do
zera i wchodzi w zielony pas tolerancji, czy robot nie przestrzeliwuje celu i czy
wybierane prędkości maleją przy celu.

In [ ]:
def plot_episode(trace, title):
    t = np.array(trace["t"])
    fig, axes = plt.subplots(1, 3, figsize=(16, 3.8))
    axes[0].plot(t, trace["x"], label="x_robot")
    axes[0].plot(t, trace["x_goal"], "r--", label="x_cel")
    axes[0].set(xlabel="czas t [s]", ylabel="pozycja [m]", title=f"{title}: pozycja")
    axes[0].legend()

    axes[1].plot(t, trace["e"], color="tab:orange", label="e(t)")
    axes[1].axhspan(-TOLERANCE, TOLERANCE, color="green", alpha=0.2, label="tolerancja")
    axes[1].axhline(0, color="k", lw=0.5)
    axes[1].set(xlabel="czas t [s]", ylabel="błąd e [m]", title=f"{title}: błąd")
    axes[1].legend()

    axes[2].step(t[1:], trace["v"], where="pre", color="tab:green", label="v(t)")
    axes[2].set(xlabel="czas t [s]", ylabel="prędkość v [m/s]", title=f"{title}: akcje")
    axes[2].legend()
    plt.tight_layout()
    plt.show()


rng_demo = np.random.default_rng(SEED)
demo_cases = {"cel po prawej": (-3.0, 3.0), "cel po lewej": (3.0, -3.0)}
traces = {}
for name, (x0, xg) in demo_cases.items():
    trace, ok = run_greedy_episode(env, encode, q_table, MAX_STEPS, rng_demo, x0, xg)
    traces[name] = trace
    print(f"{name}: start x={x0:+.1f} m, cel x={xg:+.1f} m -> "
          f"{'sukces' if ok else 'brak sukcesu'} po {len(trace['v'])} krokach ({trace['t'][-1]:.1f} s), "
          f"końcowy błąd {trace['e'][-1]:+.3f} m")
    plot_episode(trace, name)

### Odtwarzanie jednego epizodu krok po kroku
Zmień `ANIMATION_CASE` na `"cel po lewej"`, aby zobaczyć drugi przypadek.
Pod animacją tabela pokazuje wartości `e`, stan dyskretny i wybraną prędkość w pierwszych krokach.

In [ ]:
ANIMATION_CASE = "cel po prawej"

trace = traces[ANIMATION_CASE]
fig, ax = plt.subplots(figsize=(8, 1.8))
robot_marker, = ax.plot([], [], "bs", ms=14, label="robot")
ax.plot([trace["x_goal"][0]], [0], "r*", ms=18, label="cel")
ax.axhline(0, color="gray", lw=1)
ax.set(xlim=(X_MIN - 0.5, X_MAX + 0.5), ylim=(-1, 1), xlabel="x [m]")
ax.set_yticks([])
ax.legend(loc="upper right")
title = ax.set_title("")


def update(frame):
    robot_marker.set_data([trace["x"][frame]], [0])
    title.set_text(f"t = {trace['t'][frame]:.1f} s, e = {trace['e'][frame]:+.2f} m")
    return robot_marker, title


anim = FuncAnimation(fig, update, frames=len(trace["x"]), interval=100, blit=False)
plt.close(fig)
HTML(anim.to_jshtml())

In [ ]:
print(" krok |  t [s] |  e [m] | stan | v [m/s]")
for k in range(min(10, len(trace["v"]))):
    print(f"{k:5d} | {trace['t'][k]:6.1f} | {trace['e'][k]:+6.2f} | {encode(trace['e'][k]):4d} | {trace['v'][k]:+5.1f}")

## 9. Eksperymenty

Każdy eksperyment uczy agenta od zera z innym ustawieniem i mierzy skuteczność w trybie
zachłannym na nowych, losowych epizodach. Skuteczność zależy też od ziarna, więc traktuj
wyniki jako tendencję, nie dokładną liczbę. Jeśli komórka trwa zbyt długo, zmniejsz
`N_EPISODES_EXP`.

In [ ]:
N_EPISODES_EXP = 3000        # liczba epizodów uczenia w każdym eksperymencie [-]
N_EVAL = 200                 # liczba epizodów testowych [-]


def run_experiment(delta_t=None, n_bins=None, speeds=None, n_episodes=None):
    """Uczy agenta z wybranymi ustawieniami; zwraca (skuteczność, średni czas [s], historia)."""
    delta_t = DELTA_T if delta_t is None else delta_t
    n_bins = N_BINS if n_bins is None else n_bins
    speeds = SPEEDS if speeds is None else np.asarray(speeds, dtype=float)
    n_episodes = N_EPISODES_EXP if n_episodes is None else n_episodes
    env_exp = make_env(delta_t, speeds)

    def encode_exp(e):
        return discretize_error(e, n_bins)

    # Przy małym Δt potrzeba więcej kroków na ten sam dystans.
    max_steps = int(np.ceil(MAX_STEPS * max(1.0, DELTA_T / delta_t)))
    q, hist = train(env_exp, encode_exp, n_bins, len(speeds), n_episodes, ALPHA, GAMMA,
                    EPSILON_START, EPSILON_END, EPSILON_MODE, max_steps, SEED)
    rate, steps = success_rate(env_exp, encode_exp, q, N_EVAL, max_steps, SEED + 1)
    return rate, steps * delta_t, hist


def show_results(title, results, xlabel):
    labels = list(results)
    fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
    for label in labels:
        hist = results[label][2]
        axes[0].plot(*moving_average(np.array(hist["success"], dtype=float), 100), label=label)
    axes[0].set(xlabel="numer epizodu [-]", ylabel="odsetek sukcesów (uczenie) [-]", title=title)
    axes[0].legend()
    axes[1].bar(labels, [results[l][0] for l in labels], color="tab:blue")
    axes[1].set(xlabel=xlabel, ylabel="skuteczność, epsilon = 0 [-]", ylim=(0, 1.05),
                title="Skuteczność po treningu")
    for i, l in enumerate(labels):
        axes[1].text(i, results[l][0] + 0.02, f"{results[l][1]:.1f} s", ha="center")
    plt.tight_layout()
    plt.show()
    print("(liczba nad słupkiem: średni czas dojazdu udanych epizodów)")

### 9a. Wpływ czasu między decyzjami `Δt`
Dla `Δt = 1,0 s` jeden krok z prędkością 1 m/s to aż 1 m. Tolerancja to 0,1 m,
więc robot będzie często przestrzeliwał cel. Dla małego `Δt` przemieszczenia są
małe, ale dojazd wymaga wielu kroków (a więc więcej decyzji do nauczenia).

In [ ]:
dt_results = {f"Δt={dt}": run_experiment(delta_t=dt) for dt in (0.05, 0.2, 1.0)}
show_results("Wpływ Δt", dt_results, "czas między decyzjami")

### 9b. Wpływ rozdzielczości dyskretyzacji
Mało przedziałów: tablica jest mała i uczy się szybko, ale przedział jest szeroki
(przy 20 przedziałach: 1 m), więc polityka jest zgrubna. Dużo przedziałów: precyzja
reprezentacji jest większa, lecz do poznania jest więcej wartości Q, więc przy tej samej
liczbie epizodów uczenie jest wolniejsze (przy 400 przedziałach skuteczność zwykle spada).
Przy tak prostym zadaniu 20 przedziałów może wystarczyć — zobacz, jak wygląda polityka
(sekcja 7) po zmianie `N_BINS` na 20.

In [ ]:
bins_results = {f"{nb} przedz.": run_experiment(n_bins=nb) for nb in (20, 100, 400)}
show_results("Wpływ liczby przedziałów", bins_results, "liczba przedziałów błędu")

### 9c. Wpływ zestawu akcji
Więcej prędkości daje większą kontrolę (np. wolny dojazd przy celu), ale zwiększa liczbę
wartości w tablicy Q. Przy zestawie `{-1, 0, 1}` krok to zawsze 0,2 m, a okno sukcesu
ma szerokość 0,2 m, więc cel łatwo „przeskoczyć". Różnice mogą być niewielkie —
porównaj też czas dojazdu (liczba nad słupkiem) i wykres `x(t)` z sekcji 8.

In [ ]:
action_sets = {"{-1, 0, 1}": [-1.0, 0.0, 1.0],
               "{-1, -0.5, 0, 0.5, 1}": [-1.0, -0.5, 0.0, 0.5, 1.0]}
actions_results = {name: run_experiment(speeds=sp) for name, sp in action_sets.items()}
show_results("Wpływ zestawu akcji", actions_results, "zestaw prędkości [m/s]")

## 10. Co jeszcze spróbować zmienić

1. `EPSILON_MODE = "const"` z `EPSILON_START = 0.3` — jak wpływa to na wykres sukcesów w trakcie
   uczenia, a jak na zachowanie w demonstracji (epsilon = 0)?
2. `GAMMA` = 0.5 i 0.99 — czy agent nadal dojeżdża do odległych celów?
3. `TOLERANCE` = 0,02 m przy `DELTA_T = 0.2` — czy cel jest jeszcze osiągalny dla najmniejszego
   kroku `0,5 · 0,2 = 0,1 m`? (Tolerancję, akcje i `Δt` trzeba dobierać razem.)
4. Zmień nagrodę na samą `-1` za krok (bez składnika `-|e|`). Czy uczenie jest wolniejsze?
   Dlaczego nagroda zależna od `e` daje „podpowiedź" na każdym kroku?
5. Zwróć uwagę na analogię do ćwiczenia: `discretize_error(e)` odpowiada `discretize_state`,
   `choose_action` — `choose_action`, `update_q_table` — `update_q_table`, a jeden obrót pętli
   `while` w `train` — jednemu wywołaniu `learning_loop` z timera ROS 2.